<a href="https://colab.research.google.com/github/hung0221/AI/blob/main/0704_Colab_LINE_Bot_with_GEMINI_Tooluse_copy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.9/818.9 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 7.4 MB/s eta 0:00:00


In [2]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [3]:
import os
from pyngrok import ngrok

In [4]:
ngrok.kill()

In [5]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://sanford-uneffusive-unfearingly.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://sanford-uneffusive-unfearingly.ngrok-free.dev


True

In [6]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

google_search_tool = Tool(
   google_search=GoogleSearch()
)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        system_instruction="你是一個中文的AI助手，請用繁體中文回答",
        tools=[google_search_tool],
        response_modalities=["TEXT"],
    )
)

In [7]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [11]:
result = stateful_query("簡介賴清德")
print(result)

賴清德（1959年10月6日出生），中華民國政治人物，民主進步黨籍，現任中華民國總統、民主進步黨主席、國家安全會議主席以及中華文化總會會長。 他曾任中華民國副總統、行政院院長、臺南市市長、立法委員及國民大會代表等多項職務。

**早年生活與學經歷**
賴清德出生於新北市萬里區的礦工家庭，兩歲時父親因礦災過世，由母親獨力撫養六個子女長大。 他先後畢業於國立臺灣大學復健醫學系、國立成功大學學士後醫學系，並取得美國哈佛大學公共衛生學院碩士學位，是臺灣少數同時具備復健、醫療和公共衛生專長的醫師。

**政治生涯**
賴清德的政治生涯始於1994年，他擔任民主進步黨籍省長候選人陳定南的臺南醫師後援會會長，自此投入公共事務。
*   **國民大會代表與立法委員時期：** 1996年，臺海飛彈危機之際，他棄醫從政，在臺南市以第一高票當選國民大會代表，並參與廢除國大代表的歷史任務。 1998年，他當選臺南市立法委員，並連任四屆（1999-2010年），期間深耕衛環社福委員會，並曾獲公民監督國會聯盟評鑑為「問政表現第一名」。
*   **臺南市市長時期：** 2010年，賴清德當選臺南縣市合併升格後的第一任直轄市市長。 他以「清廉勤政、傳承創新」為競選主軸，並在2014年競選連任時，創下臺南選舉史上及臺灣解嚴後縣市首長選舉的最高得票率紀錄。 在市長任內，他積極推動城市發展、災害防治（如治水工程獲得顯著經費投入）及社會福利，並強化城市外交。
*   **行政院院長時期：** 2017年9月，賴清德接任行政院院長，籌組「做實事」內閣，推動加薪、減稅、改善投資環境等政策，並擘劃「文化臺灣、綠能矽島、智慧國家、公義社會、幸福家園」等施政目標。
*   **副總統時期：** 2020年，賴清德與蔡英文搭檔參選第15任總統、副總統，並成功當選副總統。 他於2023年1月兼任民主進步黨主席。
*   **總統時期：** 2024年1月，賴清德當選中華民國第16任總統，成為中華民國憲政史上首位以副總統身分參選並成功當選的政治人物，也是第二位歷任行政院院長、副總統及總統職務的政治人物。 他於同年5月20日宣誓就職。

**施政理念與展望**
賴清德總統強調「民主和平的臺灣」、「創新繁榮的臺灣」以及「公義永續的臺灣」三大施政理念。 他致力於維護臺海和平現狀，透過「民主和平四大支柱」行動方案展現決心。 在

In [12]:
result2 = stateful_query("近期新聞？")
print(result2)

以下是賴清德總統及臺灣近期的一些重要新聞與施政重點：

**賴清德總統的近期活動與談話：**

*   **強調2026年是關鍵一年**：賴清德總統在2026年1月5日接見進出口商業同業公會全國聯合會會員時表示，去（2025）年臺灣出口規模接近5,785億美元，創下歷史新高，經濟成長率達7.37%。他指出，2026年是關鍵的一年，政府將在既有基礎上提升產業實力，並期盼朝野合作，凝聚更大力量，共同面對全球供應鏈重組、美國關稅政策及地緣政治風險等挑戰。
*   **發表2026年新年談話**：賴清德總統於2026年元旦發表新年談話，主題為「韌性之島，希望之光」。他回顧2025年臺灣克服多項挑戰，展現團結與韌性，並展望2026年，強調政府將持續與人民並肩，凝聚社會力量、強化國家韌性。他也期盼2025年的朝野僵局不會延續到2026年，攸關國家生存發展的建設不會再因杯葛而停滯，並強調將積極促成朝野和諧與合作。
*   **承諾調升基本工資**：賴清德總統於2026年1月4日出席國際青年商會活動時表示，政府承諾2026年將再度調升基本工資，目標是讓基本工資跨越新臺幣3萬元大關，以照顧基層勞工。
*   **因應台美關稅挑戰**：賴總統指出，台美關稅談判已進入最後階段，部分關稅已獲調降。行政院也編列新臺幣930億元專案預算，從人才培育、金融支持、穩定就業及拓展國際通路等四方向，協助受關稅影響的中小微型企業。
*   **推動長照3.0與社會福利**：賴清德總統在2026年1月2日表示，政府今年啟動長照3.0，加強對長輩的照顧，投入預算1,200億元，包括推動機構式長照服務、放寬長照服務對象、增加長照據點等。他也呼籲立法院早日審議115年度中央政府總預算，避免影響民眾權益。
*   **接見歐洲議會議員團**：賴清德總統於2026年1月6日接見「歐洲議會議員團」時表示，面對威權主義的複合式挑戰，民主國家唯有團結合作，才能捍衛自由民主價值。

**臺灣整體近期焦點：**

*   **經濟表現與展望**：元大投顧預估2026年臺灣GDP（國內生產毛額）成長率上看4.61%，高於市場共識值，主要受惠於AI產業相關需求帶動出口強勁成長，以及民間消費的支撐。
*   **科技創新展現國際實力**：國科會TTA號召57家新創公司攜手83家供應鏈夥伴，將前進美國消費性電子展（CES 2026）

In [ ]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        if text.startswith('AI '):
            prompt = text[3:]
            reply_text = stateful_query(prompt)
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit
